## From Clicks to Loyalty:

Mining Customer Journey Patterns in Real E-Commerce Data

## Required Preprocessing

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.simplefilter("ignore")

## Load and Concatenate

In [2]:
file_path = '/content/online_retail_II.xlsx'
df1 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
df2 = pd.read_excel(file_path, sheet_name='Year 2010-2011')
df = pd.concat([df1, df2], ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Cleaning

In [3]:
# Rename columns
df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

# Remove cancelled invoices
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove null Customer IDs
df = df.dropna(subset=['CustomerID'])

# Remove invalid Quantity/Price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Restrict to UK
df = df[df['Country'] == 'United Kingdom']

# Convert types
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.shape

(725250, 8)

### Basic Stats & Report

In [4]:
total_invoices = df['Invoice'].nunique()
unique_customers = df['CustomerID'].nunique()
unique_products = df['StockCode'].nunique()
date_range = (df['InvoiceDate'].min(), df['InvoiceDate'].max())

# Basket size
basket_size = df.groupby('Invoice')['StockCode'].nunique().mean()

print("Total invoices:", total_invoices)
print("Unique customers:", unique_customers)
print("Unique products:", unique_products)
print("Date range:", date_range)
print("Avg basket size:", basket_size)

Total invoices: 33541
Unique customers: 5350
Unique products: 4616
Date range: (Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:49:00'))
Avg basket size: 20.571032467726067


## Part A: FP-Growth — Efficient Co-Purchase Mining

In [5]:
df['StockCode'] = df['StockCode'].astype(str)
# One transaction per invoice
transactions = df.groupby('Invoice')['StockCode'].apply(lambda x: list(set(x))).tolist()

### A1

In [6]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

# Frequent itemsets
freq_items = fpgrowth(df_encoded, min_support=0.02, use_colnames=True)

# Rules
rules = association_rules(freq_items, metric="confidence", min_threshold=0.5)

freq_items.head(), rules.head()

(    support itemsets
 0  0.051221  (21232)
 1  0.024299  (21523)
 2  0.074655  (84879)
 3  0.052712  (21754)
 4  0.044274  (21181),
   antecedents consequents  antecedent support  consequent support   support  \
 0     (21755)     (21754)            0.043111            0.052712  0.022778   
 1     (21733)    (85123A)            0.046212            0.140366  0.032676   
 2     (20726)     (20725)            0.039713            0.068096  0.021943   
 3     (84991)     (21212)            0.047256            0.063027  0.024269   
 4     (21977)     (21212)            0.041800            0.063027  0.022092   
 
    confidence       lift  representativity  leverage  conviction  \
 0    0.528354  10.023487               1.0  0.020506    2.008474   
 1    0.707097   5.037539               1.0  0.026190    2.934875   
 2    0.552553   8.114346               1.0  0.019239    2.082712   
 3    0.513565   8.148284               1.0  0.021290    1.926202   
 4    0.528531   8.385737               

### A2

In [7]:
freq_items['length'] = freq_items['itemsets'].apply(len)
freq_items['length'].value_counts().sort_index()
top15 = rules.sort_values(by='lift', ascending=False).head(15)

top15[['antecedents','consequents','support','confidence','lift']]

,antecedents,consequents,support,confidence,lift
5,(21231),(21232),0.023613,0.721969,14.095205
9,(82494L),(82482),0.028979,0.565445,11.772558
10,(82482),(82494L),0.028979,0.603352,11.772558
0,(21755),(21754),0.022778,0.528354,10.023487
4,(21977),(21212),0.022092,0.528531,8.385737
3,(84991),(21212),0.024269,0.513565,8.148284
2,(20726),(20725),0.021943,0.552553,8.114346
12,(22384),(20725),0.027757,0.548292,8.051780
7,(20727),(20725),0.028681,0.505783,7.427531
11,(85099F),(85099B),0.027250,0.632526,7.105008


## PART A3 — Product Bundles

In [8]:
strong_rules = rules[(rules['confidence'] >= 0.5) & (rules['lift'] > 2)]

strong_rules = strong_rules.sort_values(by='lift', ascending=False)

strong_rules[['antecedents','consequents','support','confidence','lift']].head(10)
for i, row in strong_rules.head(5).iterrows():
    print("Bundle:", list(row['antecedents']), "+", list(row['consequents']))

Bundle: ['21231'] + ['21232']
Bundle: ['82494L'] + ['82482']
Bundle: ['82482'] + ['82494L']
Bundle: ['21755'] + ['21754']
Bundle: ['21977'] + ['21212']


Using the association rules generated via FP-Growth, we identify strong item relationships with high confidence and lift. These rules indicate products that are frequently co-purchased and are suitable for bundling into gift sets.
### **Bundle 1**
Items: 21231 + 21232
Rule: {21231} → {21232}
Support: 0.0236
Confidence: 0.7219
Lift: 14.10

Explanation:
This rule shows a very strong association between the two items, with a high lift value (>14), indicating that customers who purchase 21231 are over 14 times more likely to also purchase 21232 compared to random chance. The high confidence (72%) further confirms this relationship. These items are highly complementary and should be bundled together as a gift set to maximize cross-selling opportunities.

### **Bundle 2**
Items: 82494L + 82482
Rule: {82494L} → {82482} (and reverse also exists)
Support: 0.0289
Confidence: 0.5654
Lift: 11.77

Explanation:
This pair shows a strong bidirectional relationship with high lift (11.77), meaning both items are frequently purchased together. The confidence values (~56–60%) indicate a consistent co-purchase pattern. This suggests that these products are functionally or thematically related and can be effectively packaged as a bundle or promotional gift set.

### Conclusion

The selected bundles are based on:

High lift (>10) → strong association beyond chance
High confidence (>50%) → reliable co-purchase behavior
Sufficient support → occurs frequently in transactions

These characteristics make them ideal candidates for bundling strategies to improve sales, customer convenience, and marketing effectiveness.

## Part B: Class Association Rules — Customer Segmentation

### RFM Computation

In [9]:
df['TotalPrice'] = df['Quantity'] * df['Price']

max_date = df['InvoiceDate'].max()

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (max_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID','Recency','Frequency','Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,325,12,77556.46
1,12608,404,1,415.79
2,12745,486,2,723.85
3,12746,540,1,254.55
4,12747,1,26,9276.54


### Segment Assignment

In [10]:
r_q = rfm['Recency'].quantile(0.75)
f_q = rfm['Frequency'].quantile(0.75)
m_q = rfm['Monetary'].quantile(0.75)

def segment(row):
    if row['Frequency'] <= 2 and row['Monetary'] < rfm['Monetary'].quantile(0.25):
        return 'One-and-Done'
    elif row['Recency'] > r_q:
        return 'At-Risk'
    elif row['Frequency'] > f_q and row['Monetary'] > m_q:
        return 'Champion'
    else:
        return 'Other'

rfm['Segment'] = rfm.apply(segment, axis=1)

rfm['Segment'].value_counts()

,count
Segment,
Other,2415
One-and-Done,1272
Champion,999
At-Risk,664


### Merge Segment Back

In [11]:
df = df.merge(rfm[['CustomerID','Segment']], on='CustomerID', how='left')

### B1

### Car prepration

In [12]:
from mlxtend.preprocessing import TransactionEncoder

transactions_car = df.groupby(['Invoice','Segment'])['StockCode'].apply(list).reset_index()

transactions_car['full'] = transactions_car.apply(
    lambda row: row['StockCode'] + [f"class={row['Segment']}"], axis=1
)

te = TransactionEncoder()
te_array = te.fit(transactions_car['full']).transform(transactions_car['full'])

df_car = pd.DataFrame(te_array, columns=te.columns_)

### Car Mining

In [13]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [14]:
freq_itemsets = fpgrowth(df_car, min_support=0.01, use_colnames=True)
rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.55)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(21232),(class=Champion),0.051221,0.602665,0.033034,0.644936,1.070139,1.0,0.002165,1.119051,0.069081,0.053208,0.106385,0.349875
1,(21523),(class=Champion),0.024299,0.602665,0.017739,0.730061,1.211388,1.0,0.003096,1.471944,0.178846,0.029118,0.320626,0.379748
2,(21871),(class=Champion),0.016368,0.602665,0.012075,0.737705,1.224070,1.0,0.002210,1.514838,0.186100,0.019894,0.339863,0.378870
3,(84879),(class=Champion),0.074655,0.602665,0.044841,0.600639,0.996638,1.0,-0.000151,0.994926,-0.003633,0.070897,-0.005100,0.337521
4,"(84879, 85123A)",(class=Champion),0.018544,0.602665,0.012433,0.670418,1.112422,1.0,0.001256,1.205571,0.102970,0.020422,0.170518,0.345524


In [15]:
def get_class(itemset):
    for item in itemset:
        if str(item).startswith("class="):
            return item
    return None

rules['class'] = rules['consequents'].apply(lambda x: get_class(list(x)))

car_rules = rules[rules['class'].notnull()]
for seg in car_rules['class'].unique():
    print("\nSegment:", seg)

    display(
        car_rules[car_rules['class'] == seg]
        .sort_values(by='confidence', ascending=False)
        .head(5)[['antecedents','class','support','confidence']]
    )


Segment: class=Champion


,antecedents,class,support,confidence
469,"(22386, 22385)",class=Champion,0.010405,0.830952
126,"(20725, 20724)",class=Champion,0.010495,0.830189
218,"(85099B, 22386, 21931)",class=Champion,0.010495,0.826291
189,"(21930, 85099B)",class=Champion,0.012194,0.824597
214,"(20725, 21931)",class=Champion,0.011180,0.824176


## B2

In [17]:
new_items = {'22386', '22385', '85099B'}

In [18]:
# Filter rules that fire
fired_rules = car_rules[
    car_rules['antecedents'].apply(lambda x: set(x).issubset(new_items))
]

# Sort by confidence (strongest first)
fired_rules = fired_rules.sort_values(by='confidence', ascending=False)

fired_rules[['antecedents', 'consequents', 'support', 'confidence']]

,antecedents,consequents,support,confidence
469,"(22386, 22385)",(class=Champion),0.010405,0.830952
470,"(85099B, 22385)",(class=Champion),0.011419,0.816631
464,"(85099B, 22386)",(class=Champion),0.023285,0.790486
468,(22385),(class=Champion),0.019856,0.773519
462,(22386),(class=Champion),0.036821,0.752132
38,(85099B),(class=Champion),0.065890,0.740121


In [19]:
# Get predicted segments from consequents
predictions = fired_rules['consequents'].apply(lambda x: list(x)[0])

print(predictions.value_counts())

consequents
class=Champion    6
Name: count, dtype: int64


In [20]:
print("\n--- Fired Rules ---")
for _, row in fired_rules.iterrows():
    print(f"Rule: {set(row['antecedents'])} → {list(row['consequents'])}")
    print(f"Confidence: {row['confidence']:.4f}, Support: {row['support']:.4f}")
    print("-"*50)


--- Fired Rules ---
Rule: {'22386', '22385'} → ['class=Champion']
Confidence: 0.8310, Support: 0.0104
--------------------------------------------------
Rule: {'85099B', '22385'} → ['class=Champion']
Confidence: 0.8166, Support: 0.0114
--------------------------------------------------
Rule: {'85099B', '22386'} → ['class=Champion']
Confidence: 0.7905, Support: 0.0233
--------------------------------------------------
Rule: {'22385'} → ['class=Champion']
Confidence: 0.7735, Support: 0.0199
--------------------------------------------------
Rule: {'22386'} → ['class=Champion']
Confidence: 0.7521, Support: 0.0368
--------------------------------------------------
Rule: {'85099B'} → ['class=Champion']
Confidence: 0.7401, Support: 0.0659
--------------------------------------------------


## PART C — SEQUENTIAL MINING

In [22]:
def map_category(desc):
    desc = str(desc).upper()

    if 'CANDLE' in desc or 'SCENT' in desc:
        return 'Candles & Fragrance'
    elif 'CHRISTMAS' in desc or 'TREE' in desc or 'DECORATION' in desc:
        return 'Seasonal & Festive'
    elif 'MUG' in desc or 'PLATE' in desc or 'KITCHEN' in desc:
        return 'Kitchen & Dining'
    elif 'BAG' in desc or 'BOX' in desc or 'STORAGE' in desc:
        return 'Storage & Packaging'
    elif 'TOY' in desc or 'GAME' in desc:
        return 'Toys & Games'
    elif 'LIGHT' in desc or 'LAMP' in desc:
        return 'Lighting & Decor'
    else:
        return 'Other'
df['Category'] = df['Description'].apply(map_category)

In [23]:
# Get invoice-level category baskets
invoice_cat = df.groupby(['CustomerID','Invoice','InvoiceDate'])['Category'].apply(set).reset_index()

# Sort invoices per customer
invoice_cat = invoice_cat.sort_values(by=['CustomerID','InvoiceDate'])

# Build sequences per customer
sequences = invoice_cat.groupby('CustomerID')['Category'].apply(list).reset_index()
sequences.head()

,CustomerID,Category
0,12346,"[{Other}, {Other}, {Other}, {Other}, {Other}, ..."
1,12608,"[{Other, Lighting & Decor}]"
2,12745,"[{Storage & Packaging, Other, Kitchen & Dining..."
3,12746,"[{Storage & Packaging, Other, Kitchen & Dining..."
4,12747,"[{Other, Lighting & Decor}, {Other, Lighting &..."


In [24]:
# Represent each sequence as a tuple of categories per invoice
sequence_data = sequences['Category'].tolist()

In [25]:
# Flatten for sequence mining
all_sequences = []

for seq in sequence_data:
    for i in range(len(seq)):
        all_sequences.append(tuple(seq[i]))
from collections import Counter

flat_items = [item for seq in sequence_data for step in seq for item in step]
item_counts = Counter(flat_items)

total_sequences = len(sequence_data)

freq_1 = {k:v/total_sequences for k,v in item_counts.items()}

In [26]:
from itertools import combinations

pair_counts = Counter()

for seq in sequence_data:
    for i in range(len(seq)-1):
        for a in seq[i]:
            for b in seq[i+1]:
                pair_counts[(a,b)] += 1

freq_2 = {k:v/total_sequences for k,v in pair_counts.items()}

In [34]:
triplet_counts = Counter()

for seq in sequence_data:
    for i in range(len(seq)-2):
        for a in seq[i]:
            for b in seq[i+1]:
                for c in seq[i+2]:
                    triplet_counts[(a,b,c)] += 1

freq_3 = {k:v/total_sequences for k,v in triplet_counts.items()}

### C1

In [29]:
# Top 10 frequent 2-step sequences
top_2 = sorted(freq_2.items(), key=lambda x: x[1], reverse=True)[:10]

df_top2 = pd.DataFrame(top_2, columns=['Sequence','Support'])
df_top2

,Sequence,Support
0,"(Other, Other)",4.847664
1,"(Storage & Packaging, Other)",3.122804
2,"(Other, Storage & Packaging)",3.064673
3,"(Storage & Packaging, Storage & Packaging)",2.372897
4,"(Lighting & Decor, Other)",2.293832
5,"(Other, Lighting & Decor)",2.249159
6,"(Other, Seasonal & Festive)",1.742056
7,"(Seasonal & Festive, Other)",1.673084
8,"(Kitchen & Dining, Other)",1.581495
9,"(Other, Kitchen & Dining)",1.511215


## C2

In [30]:
at_risk_customers = rfm[rfm['Segment'] == 'At-Risk']['CustomerID']

df_atrisk = df[df['CustomerID'].isin(at_risk_customers)]

In [31]:
invoice_cat_atrisk = df_atrisk.groupby(['CustomerID','Invoice','InvoiceDate'])['Category'].apply(set).reset_index()
invoice_cat_atrisk = invoice_cat_atrisk.sort_values(by=['CustomerID','InvoiceDate'])

sequences_atrisk = invoice_cat_atrisk.groupby('CustomerID')['Category'].apply(list).reset_index()

sequence_data_atrisk = sequences_atrisk['Category'].tolist()
from collections import Counter

pair_counts_atrisk = Counter()

for seq in sequence_data_atrisk:
    for i in range(len(seq)-1):
        for a in seq[i]:
            for b in seq[i+1]:
                pair_counts_atrisk[(a,b)] += 1

total_atrisk = len(sequence_data_atrisk)

freq_2_atrisk = {k:v/total_atrisk for k,v in pair_counts_atrisk.items()}

### C3

In [32]:
# Example trigger rules from top sequences
top_triggers = sorted(freq_2_atrisk.items(), key=lambda x: x[1], reverse=True)[:3]

for seq, sup in top_triggers:
    print("IF customer buys:", seq[0], "→ then", seq[1])
    print("Support:", sup)

IF customer buys: Other → then Other
Support: 1.7198795180722892
IF customer buys: Storage & Packaging → then Other
Support: 1.1069277108433735
IF customer buys: Other → then Storage & Packaging
Support: 1.0286144578313252


In [33]:
# Approximate monthly flagged customers
flagged = df_atrisk.groupby('CustomerID').size().count()
print("Approx flagged customers:", flagged)

Approx flagged customers: 664
